# CAR-T compact reformulation — FIFO rule in the Baseline

Same compact formulation as `CART_reformulation.ipynb` (branch `reformulation`), with one change:
the **Baseline** gets a first-in-first-out rule. The **Patient-aware** model is unchanged and has no FIFO.

**Variable.** `X[p, k] = 1` if patient *p* uses plan `k = (m, j1, j2, t)`: facility *m*, outbound mode *j1*,
return mode *j2*, manufacturing start day *t*.

**Plan coefficients** (precomputed for every admissible plan)

| quantity | formula |
|---|---|
| arrival | `a_p + T_LS + TT1[c(p), m, j1]` |
| wait | `t − arrival` |
| delivery | `t + T_MFE + T_QC + TT2[m, h(p), j2]` |
| cost | `CVM[m] + TT1·U1[c,m,j1] + TT2·U2[m,h,j2] + HC·wait` |
| riskD, riskU | `pi[g(p), delivery − a_p, D]`, `pi[g(p), delivery − a_p, U]` |

A plan is admissible when `t ≥ arrival`, `delivery − a_p ≤ ND` and `delivery ≤ NT`.

**Constraints (both models)**
- (R1) one plan per patient: `Σ_k X[p,k] = 1`.
- (R2) capacity: for every facility *m* and day *t*, plans at *m* starting in `[t − T_MFE + 1, t]` ≤ `FCAP[m]`.

**FIFO (Baseline only).** Order: *p* before *q* if `a_p < a_q`, or `a_p = a_q` and *p* has the smaller index.
With the linear expressions `z[p,m] = Σ_{k at m} X[p,k]` and `S[p] = Σ_k t(k)·X[p,k]`, for every facility *m*
and every ordered pair *p* before *q*:

`S[p] ≤ S[q] + NT · (2 − z[p,m] − z[q,m])`

**Objectives.** Baseline: `min Σ X·cost` (+ FIFO). Patient-aware: `min Σ X·cost + 150,000 · Σ X·riskD` (no FIFO).

In [1]:
import os, re, time, itertools
import numpy as np
import pandas as pd
import openpyxl
import pyomo.environ as pyo

OUT_DIR = "results_reformulation_fifo"
REF_DIR = "reference_no_fifo"            # results of CART_reformulation.ipynb (no FIFO); read only
os.makedirs(os.path.join(OUT_DIR, "plans"), exist_ok=True)
pd.set_option("display.width", 260, "display.max_columns", 40, "display.max_rows", 200)

LAMBDA_D, LAMBDA_U = 150_000.0, 0.0
CLASS_COLUMN = "g(p) at 30%"             # Patients column I
SCENARIOS = {
    "Houston 10 / T_MFE 7": "CART_instance_30patients.xlsx",
    "Houston 4 / T_MFE 7":  "CART_instance_30patients_cap4.xlsx",
    "Houston 2 / T_MFE 7":  "CART_instance_30patients_cap2.xlsx",
    "Houston 4 / T_MFE 14": "CART_instance_30patients_cap4_tmfe14.xlsx",
    "Houston 2 / T_MFE 14": "CART_instance_30patients_cap2_tmfe14.xlsx",
    "Houston 10 / T_MFE 5": "CART_instance_30patients_cap10_tmfe5.xlsx",
}
MODELS = ["Baseline", "Patient-aware"]
slug = lambda s: re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
pnum = lambda p: int(p[1:])

## 1. Data loader (workbook → plain Python data, nothing hard-coded)

In [2]:
def load_instance(path):
    wb = openpyxl.load_workbook(path, data_only=True)

    def table(sheet, first_row, ncols, prefix):
        out = []
        for r in wb[sheet].iter_rows(min_row=first_row, max_col=ncols, values_only=True):
            if not (isinstance(r[0], str) and re.fullmatch(prefix + r"\d+", r[0])):
                break
            out.append(r)
        return out

    settings = {str(lab): val for lab, val, *_ in wb["Settings"].iter_rows(min_row=4, max_col=3, values_only=True) if lab}
    get = lambda start: next(v for k, v in settings.items() if k.startswith(start))
    d = dict(path=path, ND=int(get("ND")), T_LS=int(get("T_LS")), T_MFE=int(get("T_MFE")), T_QC=int(get("T_QC")),
             HC=float(get("HC")), NT=int(get("Horizon NT")))

    fac = table("Facilities", 4, 5, "m")
    d["M"] = [r[0] for r in fac]
    d["M_CITY"] = {r[0]: r[1] for r in fac}
    d["FCAP"] = {r[0]: int(r[3]) for r in fac}
    d["CVM"] = {r[0]: float(r[4]) for r in fac}

    sites = table("Sites", 4, 5, "c")
    d["PAIRED_H"] = {r[0]: r[3] for r in sites}                      # MAP[c,h] = 1 iff h is the paired hospital
    d["SITE_CITY"] = {r[0]: r[1] for r in sites}

    out_rows = table("Transport_out", 4, 7, "c")                      # (c, m, j): Days = col F, U = col G
    ret_rows = table("Transport_return", 4, 7, "m")                   # (m, h, j)
    d["TT1"] = {(r[0], r[1], r[2]): int(r[5]) for r in out_rows}
    d["U1"] = {(r[0], r[1], r[2]): float(r[6]) for r in out_rows}
    d["TT2"] = {(r[0], r[1], r[2]): int(r[5]) for r in ret_rows}
    d["U2"] = {(r[0], r[1], r[2]): float(r[6]) for r in ret_rows}
    d["J"] = sorted({k[2] for k in d["TT1"]} | {k[2] for k in d["TT2"]})
    d["MODE"] = {"j1": "fast", "j2": "slow"}                          # Modes sheet: U fast (j1), U slow (j2)

    header = [c.value for c in wb["Patients"][4]]
    gcol = header.index(CLASS_COLUMN)
    assert gcol == 8, "class column is expected in column I"
    pats = table("Patients", 5, 9, "p")
    d["P"] = [r[0] for r in pats]
    d["A"] = {r[0]: int(r[1]) for r in pats}                          # collection day a_p
    d["SITE"] = {r[0]: r[2] for r in pats}
    d["G"] = {r[0]: r[gcol] for r in pats}                            # class at collection
    d["HOSP"] = {r[0]: d["PAIRED_H"][r[2]] for r in pats}             # h(p)

    q = {str(r[0]).split(":")[0]: float(r[1]) for r in wb["Clinical"].iter_rows(min_row=4, max_row=6, max_col=2, values_only=True)}
    Pm = np.array([[1 - q["q_RU"] - q["q_RD"], q["q_RU"], q["q_RD"]],
                   [0.0, 1 - q["q_UD"], q["q_UD"]],
                   [0.0, 0.0, 1.0]])                                  # eq. (21), order (R, U, D)
    pi, Pk = {}, np.eye(3)
    for tau in range(d["NT"] + 1):
        for gi, g in enumerate("RU"):
            for si, s in enumerate("RUD"):
                pi[g, tau, s] = float(Pk[gi, si])
        Pk = Pk @ Pm
    d["PI"] = pi
    dev = 0.0                                                         # cross-check with the Pi sheet (1e-6)
    for r in wb["Pi"].iter_rows(min_row=5, max_col=7, values_only=True):
        if not isinstance(r[0], (int, float)):
            break
        if int(r[0]) <= d["NT"]:
            for (g, s), v in zip([("R", "R"), ("R", "U"), ("R", "D"), ("U", "R"), ("U", "U"), ("U", "D")], r[1:7]):
                dev = max(dev, abs(pi[g, int(r[0]), s] - v))
    assert dev <= 1e-6, f"pi does not match the Pi sheet ({dev})"
    # FIFO order: collection day, then patient index
    d["ORDER"] = sorted(d["P"], key=lambda p: (d["A"][p], pnum(p)))
    return d


inst = {name: load_instance(wbk) for name, wbk in SCENARIOS.items()}
print(pd.DataFrame({name: dict(workbook=d["path"], ND=d["ND"], NT=d["NT"], T_LS=d["T_LS"], T_MFE=d["T_MFE"], T_QC=d["T_QC"],
                               HC=d["HC"], lines=" / ".join(f"{d['M_CITY'][m].split(',')[0]} {d['FCAP'][m]}" for m in d["M"]),
                               patients=len(d["P"]), urgent=sum(g == "U" for g in d["G"].values()))
                    for name, d in inst.items()}).T.to_string())

                                                       workbook  ND  NT T_LS T_MFE T_QC      HC                                         lines patients urgent
Houston 10 / T_MFE 7              CART_instance_30patients.xlsx  40  52    1     7    7  177.22  Philadelphia 4 / Houston 10 / Los Angeles 10       30      9
Houston 4 / T_MFE 7          CART_instance_30patients_cap4.xlsx  40  52    1     7    7  177.22   Philadelphia 4 / Houston 4 / Los Angeles 10       30      9
Houston 2 / T_MFE 7          CART_instance_30patients_cap2.xlsx  40  52    1     7    7  177.22   Philadelphia 4 / Houston 2 / Los Angeles 10       30      9
Houston 4 / T_MFE 14  CART_instance_30patients_cap4_tmfe14.xlsx  40  52    1    14    7  177.22   Philadelphia 4 / Houston 4 / Los Angeles 10       30      9
Houston 2 / T_MFE 14  CART_instance_30patients_cap2_tmfe14.xlsx  40  52    1    14    7  177.22   Philadelphia 4 / Houston 2 / Los Angeles 10       30      9
Houston 10 / T_MFE 5  CART_instance_30patients_cap10

## 2. Plans and model

In [3]:
def enumerate_plans(d):
    """All admissible (p, m, j1, j2, t) with their precomputed coefficients."""
    plans = {}
    for p in d["P"]:
        c, h, a, g = d["SITE"][p], d["HOSP"][p], d["A"][p], d["G"][p]
        for m, j1, j2 in itertools.product(d["M"], d["J"], d["J"]):
            tt1, tt2 = d["TT1"][c, m, j1], d["TT2"][m, h, j2]
            arrival = a + d["T_LS"] + tt1
            for t in range(arrival, d["NT"] + 1):                                  # t >= arrival
                delivery = t + d["T_MFE"] + d["T_QC"] + tt2
                if delivery - a > d["ND"] or delivery > d["NT"]:                   # TRT <= ND, inside the horizon
                    break
                wait = t - arrival
                plans[p, m, j1, j2, t] = dict(
                    arrival=arrival, wait=wait, delivery=delivery, TRT=delivery - a,
                    cost=d["CVM"][m] + tt1 * d["U1"][c, m, j1] + tt2 * d["U2"][m, h, j2] + d["HC"] * wait,
                    riskD=d["PI"][g, delivery - a, "D"], riskU=d["PI"][g, delivery - a, "U"])
    return plans


def build_model(d, plans, fifo):
    mdl = pyo.ConcreteModel()
    mdl.K = pyo.Set(initialize=list(plans), dimen=5, ordered=True)
    mdl.X = pyo.Var(mdl.K, domain=pyo.Binary)
    by_p, by_mt = {}, {}
    for k in plans:
        by_p.setdefault(k[0], []).append(k)
        by_mt.setdefault((k[1], k[4]), []).append(k)
    # (R1) one plan per patient
    mdl.R1 = pyo.Constraint(d["P"], rule=lambda mdl, p: sum(mdl.X[k] for k in by_p[p]) == 1)
    # (R2) capacity: plans started in [t - T_MFE + 1, t] occupy a line on day t
    def r2(mdl, m, t):
        ks = [k for tt in range(max(1, t - d["T_MFE"] + 1), t + 1) for k in by_mt.get((m, tt), [])]
        return sum(mdl.X[k] for k in ks) <= d["FCAP"][m] if ks else pyo.Constraint.Skip
    mdl.R2 = pyo.Constraint(d["M"], range(1, d["NT"] + 1), rule=r2)
    # FIFO (Baseline only): z and S are linear expressions in X, not new variables
    mdl.FIFO = pyo.ConstraintList()
    if fifo:
        z = {(p, m): sum(mdl.X[k] for k in by_p[p] if k[1] == m) for p in d["P"] for m in d["M"]}
        S = {p: sum(k[4] * mdl.X[k] for k in by_p[p]) for p in d["P"]}
        order = d["ORDER"]
        for m in d["M"]:
            for i, p in enumerate(order):
                for q in order[i + 1:]:
                    mdl.FIFO.add(S[p] <= S[q] + d["NT"] * (2 - z[p, m] - z[q, m]))
    mdl.cost = pyo.Expression(expr=sum(plans[k]["cost"] * mdl.X[k] for k in plans))
    mdl.ED = pyo.Expression(expr=sum(plans[k]["riskD"] * mdl.X[k] for k in plans))
    mdl.EU = pyo.Expression(expr=sum(plans[k]["riskU"] * mdl.X[k] for k in plans))
    # objectives; only one is active at a time
    mdl.obj_Baseline = pyo.Objective(expr=mdl.cost, sense=pyo.minimize)
    mdl.obj_Patient_aware = pyo.Objective(expr=mdl.cost + LAMBDA_D * mdl.ED + LAMBDA_U * mdl.EU, sense=pyo.minimize)
    mdl.obj_minED = pyo.Objective(expr=mdl.ED, sense=pyo.minimize)   # used only by the residual-tie diagnostic
    mdl.obj_maxED = pyo.Objective(expr=mdl.ED, sense=pyo.maximize)
    mdl.cap_rhs = pyo.Param(mutable=True, initialize=0.0)
    mdl.cost_cap = pyo.Constraint(expr=mdl.cost <= mdl.cap_rhs)
    mdl.cost_cap.deactivate()
    for o in mdl.component_objects(pyo.Objective):
        o.deactivate()
    return mdl


def solve(mdl, objective, cost_cap=None):
    for o in mdl.component_objects(pyo.Objective):
        o.deactivate()
    getattr(mdl, objective).activate()
    if cost_cap is None:
        mdl.cost_cap.deactivate()
    else:
        mdl.cap_rhs.set_value(cost_cap)
        mdl.cost_cap.activate()
    opt = pyo.SolverFactory("appsi_highs")
    opt.options.update({"mip_rel_gap": 0.0, "mip_abs_gap": 1e-9, "threads": 4, "random_seed": 0, "time_limit": 1800})
    t0 = time.time()
    try:
        res = opt.solve(mdl, tee=False)
    except RuntimeError as e:                    # HiGHS returned no solution
        return "infeasible", time.time() - t0, str(e).splitlines()[0]
    tc = res.solver.termination_condition
    return ("optimal" if tc == pyo.TerminationCondition.optimal else str(tc)), time.time() - t0, str(tc)

## 3. Solve: Baseline with FIFO, Patient-aware without FIFO; verify every solution

In [4]:
def plan_table(d, plans, mdl):
    rows = []
    for k in plans:
        if pyo.value(mdl.X[k]) > 0.5:
            p, m, j1, j2, t = k
            a = plans[k]
            rows.append({"p": p, "class": d["G"][p], "site": d["SITE"][p], "site city": d["SITE_CITY"][d["SITE"][p]],
                         "hospital": d["HOSP"][p], "facility": m, "out mode": d["MODE"][j1], "return mode": d["MODE"][j2],
                         "collection day": d["A"][p], "arrival": a["arrival"], "start": t, "delivery": a["delivery"],
                         "TRT": a["TRT"], "W_p": a["wait"], "pi_D at delivery": a["riskD"], "pi_U at delivery": a["riskU"],
                         "plan cost": a["cost"], "_j1": j1, "_j2": j2})
    return pd.DataFrame(rows).sort_values("p", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)


def fifo_violations(d, tab):
    """Pairs (p, q) at the same facility with p before q in FIFO order but start(p) > start(q)."""
    bad = []
    for m in d["M"]:
        sub = tab[tab.facility == m].copy()
        sub["_ord"] = sub.p.map({p: i for i, p in enumerate(d["ORDER"])})
        sub = sub.sort_values("_ord")
        starts, ps = sub.start.tolist(), sub.p.tolist()
        for i in range(len(ps)):
            for j in range(i + 1, len(ps)):
                if starts[i] > starts[j]:
                    bad.append((m, ps[i], ps[j], starts[i], starts[j]))
    return bad


def verify(d, plans, tab, mdl, with_fifo):
    """Checks recomputed from the raw data and the chosen plans."""
    out = []
    out.append(("one plan per patient", sorted(tab.p, key=pnum) == sorted(d["P"], key=pnum) and tab.p.is_unique))
    keys = [(r.p, r.facility, r._j1, r._j2, int(r.start)) for _, r in tab.iterrows()]
    out.append(("every chosen plan is admissible", all(k in plans for k in keys)))
    load = {(m, t): int(((tab.facility == m) & (tab.start >= t - d["T_MFE"] + 1) & (tab.start <= t)).sum())
            for m in d["M"] for t in range(1, d["NT"] + 1)}
    out.append(("capacity respected for every m and t", all(load[m, t] <= d["FCAP"][m] for m, t in load)))
    out.append((f"TRT <= ND ({d['ND']}) for every patient", bool((tab.TRT <= d["ND"]).all())))
    tt1 = tab.apply(lambda r: d["TT1"][r.site, r.facility, r._j1], axis=1)
    tt2 = tab.apply(lambda r: d["TT2"][r.facility, r.hospital, r._j2], axis=1)
    out.append(("timing recomputed from route/modes (arrival, start >= arrival, delivery, TRT, horizon)",
                bool(((tab.arrival == tab["collection day"] + d["T_LS"] + tt1) & (tab.start >= tab.arrival)
                      & (tab.delivery == tab.start + d["T_MFE"] + d["T_QC"] + tt2)
                      & (tab.TRT == d["T_LS"] + tt1 + tab.W_p + d["T_MFE"] + d["T_QC"] + tt2)
                      & (tab.delivery <= d["NT"])).all())))
    ed = sum(d["PI"][r["class"], r.delivery - r["collection day"], "D"] for _, r in tab.iterrows())
    out.append(("E_D recomputed from the chosen plans = solver value (1e-6)", abs(ed - pyo.value(mdl.ED)) <= 1e-6))
    u1 = tab.apply(lambda r: d["U1"][r.site, r.facility, r._j1], axis=1)
    u2 = tab.apply(lambda r: d["U2"][r.facility, r.hospital, r._j2], axis=1)
    cost = (tab.facility.map(d["CVM"]) + tt1 * u1 + tt2 * u2 + d["HC"] * tab.W_p).sum()
    out.append(("total cost recomputed from the chosen plans = solver value", abs(cost - pyo.value(mdl.cost)) <= 1e-4))
    if with_fifo:
        out.append(("FIFO: at each facility, start days non-decreasing in (collection day, index) order",
                    len(fifo_violations(d, tab)) == 0))
    return out


RES, PLANS, VERIF, SIZE = {}, {}, {}, {}
for name, d in inst.items():
    plans = enumerate_plans(d)
    print(f"\n{name}: {len(plans):,} admissible plans")
    for which in MODELS:
        fifo = which == "Baseline"
        t0 = time.time()
        mdl = build_model(d, plans, fifo=fifo)
        build_s = time.time() - t0
        SIZE[name, which] = dict(variables=len(plans), constraints=len(mdl.R1) + len(mdl.R2) + len(mdl.FIFO),
                                 fifo_constraints=len(mdl.FIFO), build_s=build_s)
        status, secs, msg = solve(mdl, "obj_" + slug(which))
        if status != "optimal":
            RES[name, which] = dict(status=status, solve_s=secs)
            print(f"  {which:<14s} {status.upper()}: {msg}  (FIFO constraints {len(mdl.FIFO):,})")
            continue
        tab = plan_table(d, plans, mdl)
        PLANS[name, which] = tab
        VERIF[name, which] = verify(d, plans, tab, mdl, with_fifo=fifo)
        tab.drop(columns=["_j1", "_j2"]).to_csv(os.path.join(OUT_DIR, "plans", f"{slug(name)}__{which}.csv"), index=False)
        r = RES[name, which] = dict(status="optimal", objective=pyo.value(getattr(mdl, "obj_" + slug(which))),
                                    cost=pyo.value(mdl.cost), E_D=pyo.value(mdl.ED), E_U=pyo.value(mdl.EU), solve_s=secs)
        if fifo:                                  # keep the FIFO Baseline model for the residual-tie diagnostic
            RES[name, which]["_model"], RES[name, which]["_plans"] = mdl, plans
        print(f"  {which:<14s} objective={r['objective']:>14,.4f}  cost={r['cost']:>12,.2f}  E_D={r['E_D']:.6f}  "
              f"FIFO constraints={len(mdl.FIFO):>5,}  [build {build_s:.1f}s, solve {secs:.2f}s]  "
              f"verification: {'PASS' if all(o for _, o in VERIF[name, which]) else 'FAIL'}")


Houston 10 / T_MFE 7: 7,908 admissible plans


  Baseline       objective=  384,646.9000  cost=  384,646.90  E_D=4.593579  FIFO constraints=1,305  [build 0.1s, solve 7.54s]  verification: PASS


  Patient-aware  objective=1,035,503.5336  cost=  393,745.96  E_D=4.278384  FIFO constraints=    0  [build 0.1s, solve 0.41s]  verification: PASS

Houston 4 / T_MFE 7: 7,908 admissible plans


  Baseline       objective=  386,896.4400  cost=  386,896.44  E_D=4.984331  FIFO constraints=1,305  [build 0.2s, solve 7.47s]  verification: PASS


  Patient-aware  objective=1,068,850.9947  cost=  403,728.80  E_D=4.434148  FIFO constraints=    0  [build 0.1s, solve 0.29s]  verification: PASS

Houston 2 / T_MFE 7: 7,908 admissible plans


  Baseline       objective=  390,014.1200  cost=  390,014.12  E_D=5.336325  FIFO constraints=1,305  [build 0.2s, solve 8.02s]  verification: PASS


... (3 outputs left out)

  Baseline       objective=  442,979.3800  cost=  442,979.38  E_D=7.158755  FIFO constraints=1,305  [build 0.2s, solve 5.49s]  verification: PASS


  Patient-aware  objective=1,468,360.0565  cost=  450,612.94  E_D=6.784981  FIFO constraints=    0  [build 0.4s, solve 0.53s]  verification: PASS

Houston 10 / T_MFE 5: 8,628 admissible plans


  Baseline       objective=  377,999.3000  cost=  377,999.30  E_D=4.209389  FIFO constraints=1,305  [build 0.2s, solve 8.31s]  verification: PASS


  Patient-aware  objective=  950,014.0857  cost=  388,177.76  E_D=3.745576  FIFO constraints=    0  [build 0.2s, solve 0.42s]  verification: PASS


## 4. Six-scenario table (Baseline with FIFO, Patient-aware without FIFO)

In [5]:
ROWS = ["E_D, Baseline", "E_D, Patient-aware", "Difference (Baseline − Patient-aware)",
        "Relative reduction (Difference ÷ E_D Baseline)", "Extra cost, Patient-aware vs Baseline (USD)",
        "Mean wait, Baseline (days)", "Max wait, Baseline (days)",
        "Mean wait of urgent patients, Baseline (days)", "Mean wait of urgent patients, Patient-aware (days)"]

six = {}
for name in SCENARIOS:
    b, a = RES[name, "Baseline"], RES[name, "Patient-aware"]
    if b["status"] != "optimal" or a["status"] != "optimal":
        six[name] = {row: ("infeasible" if "Baseline" in row and b["status"] != "optimal" else np.nan) for row in ROWS}
        continue
    tb, ta = PLANS[name, "Baseline"], PLANS[name, "Patient-aware"]
    six[name] = dict(zip(ROWS, [b["E_D"], a["E_D"], b["E_D"] - a["E_D"], (b["E_D"] - a["E_D"]) / b["E_D"],
                                a["cost"] - b["cost"], tb.W_p.mean(), tb.W_p.max(),
                                tb.loc[tb["class"] == "U", "W_p"].mean(), ta.loc[ta["class"] == "U", "W_p"].mean()]))
six = pd.DataFrame(six).loc[ROWS]
six.to_csv(os.path.join(OUT_DIR, "six_scenarios_fifo.csv"), index_label="metric")
print(six.to_string(float_format=lambda x: f"{x:,.4f}"))

                                                    Houston 10 / T_MFE 7  Houston 4 / T_MFE 7  Houston 2 / T_MFE 7  Houston 4 / T_MFE 14  Houston 2 / T_MFE 14  Houston 10 / T_MFE 5
E_D, Baseline                                                     4.5936               4.9843               5.3363                6.8409                7.1588                4.2094
E_D, Patient-aware                                                4.2784               4.4341               4.5644                6.5381                6.7850                3.7456
Difference (Baseline − Patient-aware)                             0.3152               0.5502               0.7719                0.3028                0.3738                0.4638
Relative reduction (Difference ÷ E_D Baseline)                    0.0686               0.1104               0.1447                0.0443                0.0522                0.1102
Extra cost, Patient-aware vs Baseline (USD)                   9,099.0600          16,832.3600  

## 5. Baseline without FIFO vs with FIFO
"Without FIFO" is read from `reference_no_fifo/` (the stored per-patient plans of `CART_reformulation.ipynb`).

In [6]:
def read_ref(name, which):
    tab = pd.read_csv(os.path.join(REF_DIR, "plans", f"{slug(name)}__{which}.csv"))
    return dict(plan=tab, cost=tab["plan cost"].sum(), E_D=tab["pi_D at delivery"].sum(),
                objective=tab["plan cost"].sum() + (LAMBDA_D * tab["pi_D at delivery"].sum() if which == "Patient-aware" else 0.0))


REF = {(name, w): read_ref(name, w) for name in SCENARIOS for w in MODELS}
ref_six = pd.read_csv(os.path.join(REF_DIR, "six_scenarios_reformulation.csv")).set_index("metric")
for name in SCENARIOS:   # the stored plans reproduce the stored six-scenario table
    assert abs(ref_six.loc["E_D, Baseline", name] - REF[name, "Baseline"]["E_D"]) < 1e-9
    assert abs(ref_six.loc["Extra cost, Patient-aware vs Baseline (USD)", name]
               - (REF[name, "Patient-aware"]["cost"] - REF[name, "Baseline"]["cost"])) < 1e-6

cmp_rows = ["Baseline cost without FIFO (USD)", "Baseline cost with FIFO (USD)", "Cost increase due to FIFO (USD)",
            "E_D, Baseline without FIFO", "E_D, Baseline with FIFO",
            "Relative reduction without FIFO", "Relative reduction with FIFO"]
cmp = {}
for name in SCENARIOS:
    r0, b, a = REF[name, "Baseline"], RES[name, "Baseline"], RES[name, "Patient-aware"]
    ok = b["status"] == "optimal"
    cmp[name] = dict(zip(cmp_rows, [r0["cost"], b["cost"] if ok else "infeasible", b["cost"] - r0["cost"] if ok else np.nan,
                                    r0["E_D"], b["E_D"] if ok else "infeasible",
                                    float(ref_six.loc["Relative reduction (Difference ÷ E_D Baseline)", name]),
                                    (b["E_D"] - a["E_D"]) / b["E_D"] if ok else np.nan]))
cmp = pd.DataFrame(cmp).loc[cmp_rows]
cmp.to_csv(os.path.join(OUT_DIR, "comparison_baseline_with_vs_without_fifo.csv"), index_label="metric")
print(cmp.to_string(float_format=lambda x: f"{x:,.4f}"))

                                  Houston 10 / T_MFE 7  Houston 4 / T_MFE 7  Houston 2 / T_MFE 7  Houston 4 / T_MFE 14  Houston 2 / T_MFE 14  Houston 10 / T_MFE 5
Baseline cost without FIFO (USD)          384,646.9000         386,896.4400         390,014.1200          422,055.3200          441,561.6200          377,999.3000
Baseline cost with FIFO (USD)             384,646.9000         386,896.4400         390,014.1200          423,295.8600          442,979.3800          377,999.3000
Cost increase due to FIFO (USD)                 0.0000              -0.0000               0.0000            1,240.5400            1,417.7600                0.0000
E_D, Baseline without FIFO                      4.6581               4.9234               5.2633                6.7800                7.0225                4.1426
E_D, Baseline with FIFO                         4.5936               4.9843               5.3363                6.8409                7.1588                4.2094
Relative reduction wit

## 6. FIFO constraints and solve time per run

In [7]:
runs = pd.DataFrame([{"scenario": name, "model": w, "FIFO": "yes" if w == "Baseline" else "no",
                      "status": RES[name, w]["status"], "variables": SIZE[name, w]["variables"],
                      "FIFO constraints": SIZE[name, w]["fifo_constraints"], "constraints (total)": SIZE[name, w]["constraints"],
                      "build time (s)": round(SIZE[name, w]["build_s"], 2), "solve time (s)": round(RES[name, w]["solve_s"], 2)}
                     for name in SCENARIOS for w in MODELS])
runs.to_csv(os.path.join(OUT_DIR, "runs_size_and_time.csv"), index=False)
print(runs.to_string(index=False))
print("\nSolve time = wall time of the solver call (Pyomo translation + HiGHS).")

            scenario         model FIFO  status  variables  FIFO constraints  constraints (total)  build time (s)  solve time (s)
Houston 10 / T_MFE 7      Baseline  yes optimal       7908              1305                 1449            0.11            7.54
Houston 10 / T_MFE 7 Patient-aware   no optimal       7908                 0                  144            0.13            0.41
 Houston 4 / T_MFE 7      Baseline  yes optimal       7908              1305                 1449            0.19            7.47
 Houston 4 / T_MFE 7 Patient-aware   no optimal       7908                 0                  144            0.10            0.29
 Houston 2 / T_MFE 7      Baseline  yes optimal       7908              1305                 1449            0.19            8.02
 Houston 2 / T_MFE 7 Patient-aware   no optimal       7908                 0                  144            0.10            0.26
Houston 4 / T_MFE 14      Baseline  yes optimal       5388              1305              

## 7. Diagnostic: ties left under FIFO
FIFO fixes the start order at each facility but allows equal start days and does not fix facility or modes,
so the FIFO Baseline can still have several same-cost plans. Range of E_D over all plans with
cost ≤ C*_FIFO·(1 + 1e-6), FIFO kept. (Extra diagnostic; not part of the tables above.)

In [8]:
tie_rows = []
for name in SCENARIOS:
    b = RES[name, "Baseline"]
    if b["status"] != "optimal":
        tie_rows.append({"scenario": name, "status": b["status"]})
        continue
    mdl, cap = b["_model"], b["cost"] * (1 + 1e-6)
    lo_s, _, _ = solve(mdl, "obj_minED", cost_cap=cap); lo = pyo.value(mdl.ED)
    hi_s, _, _ = solve(mdl, "obj_maxED", cost_cap=cap); hi = pyo.value(mdl.ED)
    assert lo_s == hi_s == "optimal"
    tie_rows.append({"scenario": name, "status": "optimal", "E_D, Baseline with FIFO (solver's plan)": b["E_D"],
                     "same-cost E_D, min": lo, "same-cost E_D, max": hi, "width": hi - lo})
ties = pd.DataFrame(tie_rows)
ties.to_csv(os.path.join(OUT_DIR, "diagnostic_residual_ties_fifo.csv"), index=False)
print(ties.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

            scenario  status  E_D, Baseline with FIFO (solver's plan)  same-cost E_D, min  same-cost E_D, max    width
Houston 10 / T_MFE 7 optimal                                 4.593579            4.593579            4.593579 0.000000
 Houston 4 / T_MFE 7 optimal                                 4.984331            4.981297            5.039957 0.058659
 Houston 2 / T_MFE 7 optimal                                 5.336325            5.302959            5.342220 0.039261
Houston 4 / T_MFE 14 optimal                                 6.840884            6.809540            6.848691 0.039150
Houston 2 / T_MFE 14 optimal                                 7.158755            7.058480            7.179660 0.121179
Houston 10 / T_MFE 5 optimal                                 4.209389            4.209389            4.243976 0.034587


## 8. Checks

In [9]:
checks = []
for name in SCENARIOS:
    a, r = RES[name, "Patient-aware"], REF[name, "Patient-aware"]
    if a["status"] != "optimal":
        checks.append((name, "Patient-aware solved", False))
        continue
    rel = abs(a["objective"] - r["objective"]) / max(1.0, abs(r["objective"]))
    checks.append((name, f"Patient-aware objective identical to results_reformulation (rel. diff {rel:.1e})", rel <= 1e-6))
    checks.append((name, f"Patient-aware E_D identical to results_reformulation (diff {a['E_D'] - r['E_D']:+.1e})",
                   abs(a["E_D"] - r["E_D"]) <= 1e-9))
    same_plan = PLANS[name, "Patient-aware"][["p", "facility", "out mode", "return mode", "start"]].reset_index(drop=True) \
        .equals(r["plan"][["p", "facility", "out mode", "return mode", "start"]].reset_index(drop=True))
    checks.append((name, "Patient-aware per-patient plan identical to results_reformulation", same_plan))
    b = RES[name, "Baseline"]
    if b["status"] != "optimal":
        print(f"[INFO] {name}: Baseline with FIFO is {b['status']} - reported, scenario continues")
        continue
    checks.append((name, f"Baseline with FIFO: cost >= Baseline cost without FIFO "
                         f"({b['cost']:,.2f} >= {REF[name, 'Baseline']['cost']:,.2f})", b["cost"] >= REF[name, "Baseline"]["cost"] - 1e-6))
for (name, which), lst in VERIF.items():
    for chk, ok in lst:
        checks.append((name, f"{which}: {chk}", ok))

for name, chk, ok in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {chk}")
pd.DataFrame([(n, c, "PASS" if ok else "FAIL") for n, c, ok in checks], columns=["scenario", "check", "result"]) \
    .to_csv(os.path.join(OUT_DIR, "checks.csv"), index=False)
n_inf = sum(1 for r in RES.values() if r["status"] != "optimal")
all_ok = all(ok for *_, ok in checks)
print(f"\nOVERALL: {'PASS' if all_ok else 'FAIL'}  ({len(checks)} checks; {n_inf} runs not solved to optimality)")
assert all_ok

[PASS] Houston 10 / T_MFE 7: Patient-aware objective identical to results_reformulation (rel. diff 2.2e-16)
[PASS] Houston 10 / T_MFE 7: Patient-aware E_D identical to results_reformulation (diff +8.9e-16)
[PASS] Houston 10 / T_MFE 7: Patient-aware per-patient plan identical to results_reformulation
[PASS] Houston 10 / T_MFE 7: Baseline with FIFO: cost >= Baseline cost without FIFO (384,646.90 >= 384,646.90)
[PASS] Houston 4 / T_MFE 7: Patient-aware objective identical to results_reformulation (rel. diff 4.4e-16)
[PASS] Houston 4 / T_MFE 7: Patient-aware E_D identical to results_reformulation (diff +2.7e-15)
[PASS] Houston 4 / T_MFE 7: Patient-aware per-patient plan identical to results_reformulation
[PASS] Houston 4 / T_MFE 7: Baseline with FIFO: cost >= Baseline cost without FIFO (386,896.44 >= 386,896.44)
[PASS] Houston 2 / T_MFE 7: Patient-aware objective identical to results_reformulation (rel. diff 4.3e-16)
[PASS] Houston 2 / T_MFE 7: Patient-aware E_D identical to results_reform